# 2 — Preprocessing and tokenisation

Every rule is a modelling decision. This notebook walks through each one on
real sentences, so the effect is visible rather than asserted.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from nmt.utils.io import project_root, read_json
from nmt.viz.style import use_style

use_style()
print("project root:", ROOT)

## Normalisation

Order matters. NFC composition first (so accents unify), then control-character
removal (before whitespace collapsing, so stripping one cannot weld two words
together), then typographic folding, then whitespace.

In [ ]:
from nmt.data.cleaning import normalise

examples = [
    "espan\u0303ol",            # decomposed n + combining tilde
    "espa\u00f1ol",             # pre-composed n-with-tilde
    "don\u2019t  stop",         # curly apostrophe + double space
    "\u201cquoted\u201d text",  # curly quotes
    "dash\u2014here",           # em dash
    "  ragged\tspacing\n",
]
for raw in examples:
    print(f"{raw!r:32s} -> {normalise(raw)!r}")

print("\nThe two spellings of 'español' are different strings:",
      "espan\u0303ol" != "espa\u00f1ol")
print("...but normalise to the same one:",
      normalise("espan\u0303ol") == normalise("espa\u00f1ol"))

### Casing and punctuation are deliberately preserved

Lowercasing would shrink the vocabulary and usually buys a point of BLEU, but
it makes the system unable to produce correctly-cased output — and the
deliverable is an application whose output a human reads. Spanish `¿` and `¡`
are grammatical signal.

## Filtering

In [ ]:
from nmt.data.cleaning import CleaningRules, clean_pairs

candidates = [
    ("Hello there.", "Hola."),
    ("", "Vac\u00edo."),
    ("1997.", "1997."),
    ("Visit http://example.com", "Visita http://example.com"),
    ("Tom", "Tom"),
    ("Yes.", "S\u00ed, lo har\u00e9 pronto."),
    ("I am very tired today",
     "Estoy muy cansado hoy porque he trabajado toda la noche sin descansar "
     "ni un solo momento de verdad"),
]
kept, report = clean_pairs(candidates, CleaningRules())

print("kept:")
for pair in kept:
    print("  ", pair)
print("\nremoved:", dict(report.removed))

Note that `("Yes.", "Sí, lo haré pronto.")` **survives** despite a 1:4 length
ratio. The ratio filter is suppressed until both sides reach four tokens,
because short sentences legitimately have very different lengths.

## Component-aware splitting

The union-find structure merges every sentence that is transitively linked to
every other, then whole components are assigned to a split.

In [ ]:
from nmt.data.splitting import SplitSizes, split_pairs

pairs = [
    ("I am tired.", "Estoy cansado."),
    ("I am tired.", "Estoy cansada."),
    ("Estoy cansado." , "I'm exhausted."),   # links back through the Spanish side
]
pairs = [(a, b) for a, b in pairs]
pairs += [(f"Sentence {i}.", f"Frase {i}.") for i in range(200)]

splits, report = split_pairs(pairs, SplitSizes(validation=0.1, test=0.1), seed=0)
print("sizes    :", report.split_pairs)
print("leakage  :", sum(report.leakage_check.values()))
print("components:", report.components)

## Tokenisation

Two tokenisers, one interface. Both are fitted on the **training split only** —
fitting on the whole corpus would leak test-set surface forms into the
vocabulary and flatter the out-of-vocabulary rate we report.

In [ ]:
from nmt.data.tokenizer import load_tokenizer
from nmt.constants import TAG_EN, TAG_ES

subword = load_tokenizer(ROOT / "artifacts" / "tokenizers" / "joint_bpe.model")
word = load_tokenizer(ROOT / "artifacts" / "tokenizers" / "word_vocab.json")

print("subword vocabulary:", f"{subword.vocab_size:,}")
print("word vocabulary   :", f"{word.vocab_size:,}")

sentence = "The internationalisation of unimaginably complicated vocabulary."
print("\nsubword:", subword.tokenize(sentence))
print("\nword   :", [word.id_to_piece(i) for i in word.encode(sentence)])

The subword model splits an unseen long word into known pieces; the word-level
model maps it to `<unk>` and loses it permanently. That is the whole
trade-off.

In [ ]:
# The reserved ids are pinned, and the direction tags survive as single tokens.
for piece in ("<pad>", "<unk>", "<s>", "</s>", TAG_EN, TAG_ES):
    print(f"{piece:8s} -> id {subword.piece_to_id(piece)}")

ids = subword.encode_source("I am tired.", TAG_ES)
print("\nencoded source:", ids)
print("as pieces     :", [subword.id_to_piece(i) for i in ids])
print("round trip    :", subword.decode(ids))

## Batching: token buckets, not fixed sentence counts

The length distribution is right-skewed, so a fixed sentence count produces
batches whose padding waste swings widely. Capping by *tokens* keeps memory per
step flat.

In [ ]:
from nmt.data.build import read_split
from nmt.data.dataset import TranslationDataset, padding_waste

pairs = read_split(ROOT / "data" / "processed" / "validation.tsv")
dataset = TranslationDataset(pairs, subword)

print(f"{len(pairs):,} pairs -> {len(dataset):,} directional examples")
waste = padding_waste(dataset, max_tokens=8192)
for key, value in waste.items():
    print(f"  {key:20s} {value:,.3f}" if isinstance(value, float) else f"  {key:20s} {value:,}")

In [ ]:
# What a collated batch actually contains.
from nmt.data.dataset import collate_batch

batch = collate_batch([dataset[i] for i in range(4)])
print("source        ", tuple(batch.source.shape))
print("decoder_input ", tuple(batch.decoder_input.shape))
print("labels        ", tuple(batch.labels.shape))
print()
print("decoder_input[0]:", [subword.id_to_piece(i) for i in batch.decoder_input[0][:8].tolist()])
print("labels[0]       :", [subword.id_to_piece(i) for i in batch.labels[0][:8].tolist()])
print("\nThe labels are the decoder input shifted left by one: position t predicts t+1.")